# Предсказание покупки: synthetic smoke-эксперимент

Исполненный автономный эксперимент на **детерминированных синтетических smoke-данных**.
Он проверяет реальный код репозитория, но не оценивает качество на исходном публичном наборе.

## tl;dr

Ниже показан фактически исполненный smoke-run: объём синтетики, выбранная на validation
модель и метрики неизменяемого synthetic test split. Эти числа нельзя переносить на реальные данные.

In [1]:
import io
import json
import sys
import tempfile
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from online_shopper.data import TARGET, split_data, validate_frame
from online_shopper.evaluate import main as evaluate_model
from online_shopper.generate_smoke_data import generate_smoke_frame
from online_shopper.train import main as train_model

SEED = 20250607
ROWS = 320
smoke_frame = generate_smoke_frame(rows=ROWS, seed=SEED)
validated_frame = validate_frame(smoke_frame)
train_frame, validation_frame, test_frame = split_data(validated_frame, seed=SEED)

temporary_directory = tempfile.TemporaryDirectory(prefix="shopper-smoke-")
run_directory = Path(temporary_directory.name)
data_path = run_directory / "smoke.csv"
artifact_path = run_directory / "model.joblib"
validation_path = run_directory / "validation.json"
metrics_path = run_directory / "test_metrics.json"
errors_path = run_directory / "test_errors.csv"
validated_frame.to_csv(data_path, index=False)

with redirect_stdout(io.StringIO()):
    train_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--report", str(validation_path), "--seed", str(SEED),
    ])
    evaluate_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--metrics", str(metrics_path), "--errors", str(errors_path),
    ])

validation_report = json.loads(validation_path.read_text(encoding="utf-8"))
test_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
error_rows = pd.read_csv(errors_path)
summary = pd.DataFrame([{
    "data": "deterministic synthetic smoke",
    "rows": len(validated_frame),
    "positive_rate": validated_frame[TARGET].mean(),
    "selected_model": test_metrics["model_name"],
    "test_pr_auc": test_metrics["pr_auc"],
    "test_roc_auc": test_metrics["roc_auc"],
    "test_f1": test_metrics["f1"],
    "test_errors": len(error_rows),
}])
summary.round(4)

,data,rows,positive_rate,selected_model,test_pr_auc,test_roc_auc,test_f1,test_errors
0,deterministic synthetic smoke,320,0.2281,gradient_boosting,0.2905,0.5524,0.2727,16


## Context & Methods


Цель — проверить полный путь `generate → validate → train → evaluate` для классификации покупки.
Настоящие функции `online_shopper.train.main` и `online_shopper.evaluate.main` вызываются с временными
файлами. Pipeline обучает imputation, scaling и encoding только на train; модель и порог выбираются на validation.


            ### Key Assumptions


- Seed `20250607`, 320 строк; генератор не воспроизводит распределение UCI.
- Разбиение 60/20/20 стратифицировано по `Revenue`.
- Бюджет контакта по умолчанию равен 20% validation; smoke-значение не является бизнес-рекомендацией.
- Test используется один раз после выбора модели и порога.

## Data

Smoke-таблица создаётся локальным генератором с фиксированным seed, затем проходит ту же
проверку схемы и то же разбиение, что и пользовательский CSV. Сетевые источники не используются.

In [2]:
display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_frame), len(validation_frame), len(test_frame)],
    "positive_rate": [
        train_frame[TARGET].mean(), validation_frame[TARGET].mean(), test_frame[TARGET].mean()
    ],
}).round(4))
display(validated_frame.head(3))

,split,rows,positive_rate
0,train,192,0.2240
1,validation,64,0.2344
2,test,64,0.2344


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,54.332402,0,51.911282,31,2153.210550,0.017062,0.244573,36.896291,0.4,Mar,4,4,8,3,Returning_Visitor,False,0
1,3,68.525875,0,8.664061,33,631.129829,0.046007,0.382399,0.000000,1.0,May,4,3,5,4,Returning_Visitor,True,0
2,1,13.241796,1,12.553728,35,1237.953420,0.048916,0.021562,6.951716,0.0,Nov,3,3,2,13,New_Visitor,False,0


## Results

Первая таблица — сравнение кандидатов на validation. Следующие результаты относятся только
к synthetic test split; таблица ошибок ограничена несколькими строками.

In [3]:
validation_table = pd.DataFrame(validation_report["models"]).T
display(validation_table[["pr_auc", "roc_auc", "precision", "recall", "f1", "selected_fraction"]]
        .sort_values("pr_auc", ascending=False).round(4))
display(pd.DataFrame({"metric": ["pr_auc", "roc_auc", "precision", "recall", "f1"],
                      "synthetic_test": [test_metrics[key] for key in
                                         ["pr_auc", "roc_auc", "precision", "recall", "f1"]]}).round(4))

,pr_auc,roc_auc,precision,recall,f1,selected_fraction
gradient_boosting,0.43581,0.608163,0.384615,0.333333,0.357143,0.203125
random_forest,0.409898,0.62585,0.307692,0.266667,0.285714,0.203125
logistic_regression,0.366913,0.536054,0.307692,0.266667,0.285714,0.203125
decision_tree,0.330754,0.634014,0.466667,0.466667,0.466667,0.234375
dummy,0.234375,0.5,0.234375,1.0,0.379747,1.0


,metric,synthetic_test
0,pr_auc,0.2905
1,roc_auc,0.5524
2,precision,0.4286
3,recall,0.2000
4,f1,0.2727


In [4]:
error_rows.head(8)

,VisitorType,Month,Revenue,score,prediction,error_type
0,Returning_Visitor,Dec,0,0.937022,1,false_positive
1,New_Visitor,Nov,0,0.890665,1,false_positive
2,Other,Nov,0,0.854226,1,false_positive
3,Returning_Visitor,Nov,0,0.839161,1,false_positive
4,Returning_Visitor,Nov,1,0.236295,0,false_negative
5,Returning_Visitor,Dec,1,0.144352,0,false_negative
6,Returning_Visitor,Nov,1,0.137501,0,false_negative
7,Returning_Visitor,Dec,1,0.128030,0,false_negative


## Takeaways

Выводы ниже сформированы из сохранённых outputs текущего запуска и относятся только к smoke-проверке.

In [5]:
print(f"- На synthetic test выбран {test_metrics['model_name']}: "
      f"PR-AUC={test_metrics['pr_auc']:.4f}, ROC-AUC={test_metrics['roc_auc']:.4f}.")
print(f"- При validation-пороге synthetic test F1={test_metrics['f1']:.4f}; "
      f"ошибочных строк в отчёте: {len(error_rows)}.")
print("- Это проверка воспроизводимости кода, а не заявленная метрика UCI Online Shoppers.")

- На synthetic test выбран gradient_boosting: PR-AUC=0.2905, ROC-AUC=0.5524.
- При validation-пороге synthetic test F1=0.2727; ошибочных строк в отчёте: 16.
- Это проверка воспроизводимости кода, а не заявленная метрика UCI Online Shoppers.
